# congkiem_run — THẦY CÓ HƠN HỌC TRÒ TRÊN ĐÚNG 600 NHÓM ĐÓ KHÔNG?

**~5 phút GPU.** Chấm 4.800 cặp bằng học trò `AITeamVN` (0,058 s/cặp), so thẳng với nhãn thầy.

### Vì sao cần lượt này — cổng cũ đặt SAI

Cổng cũ: *"thầy xếp gold hạng 1 ở ≥ 0,60"*. Kết quả **0,5417** → trượt.
Nhưng con số đó **không có mốc đối chiếu**: negative ở đây là top-20 BM25 trong kho pháp
luật, **29% được thầy chấm > 0,5** vì chúng liên quan thật. Gold không đứng nhất giữa 8 văn
bản cùng chủ đề là bình thường, không có nghĩa thầy dốt.

Thứ thật sự quyết định: **thầy có biết điều gì mà học trò không biết không.**

### CỔNG MỚI (đặt TRƯỚC khi chạy)

> **`gold@1` của thầy − `gold@1` của học trò ≥ +0,08** trên cùng 600 nhóm.
>
> Đạt → nhãn có thông tin học trò chưa có → chạy `chungcat_run.ipynb`.
> Không đạt → thầy chỉ biết đúng thứ học trò đã biết → **chưng cất vô nghĩa, dừng hẳn.**


In [ ]:
!pip install -q -U sentence-transformers


In [ ]:
import os, sys, json, glob, time
import torch
INPUT_DIR = next(p for p in ("/kaggle/input/project-ir",
                             "/kaggle/input/datasets/locdovan211/project-ir")
                 if os.path.isdir(p))
CTX_DIR = next(p for p in (f"{INPUT_DIR}/selected-contexts/selected-contexts",
                           f"{INPUT_DIR}/selected-contexts")
               if os.path.isdir(p) and any(f.startswith("context_") for f in os.listdir(p)))
OUT = "/kaggle/working/outputs"; os.makedirs(OUT, exist_ok=True)
sys.path.append(INPUT_DIR)

def tim(ten):
    p = glob.glob(f"{INPUT_DIR}/**/{ten}", recursive=True)
    assert p, f"KHÔNG THẤY {ten} — upload rồi chạy lại"
    return sorted(p, key=len)[0]

import deep_chunk as DC
from rerank import load_reranker
DC.MERGE_CHARS = 1800
EXC = 900                                  # PHẢI khớp lượt sinh nhãn

nhan  = json.load(open(tim("nhan_thay.json"), encoding="utf-8"))
train = json.load(open(tim("train.json"), encoding="utf-8"))
nhom  = {q: list(e) for q, e in nhan.items() if len(e) == 8}
print(f"{len(nhom)} nhóm × 8 = {len(nhom)*8:,} cặp")

t0 = time.time(); cap, chiso = [], []
for i, (q, docs) in enumerate(nhom.items(), 1):
    qs = train[q]["question"]
    for d in docs:
        cap.append([qs, (DC.pick_chunks(qs, CTX_DIR, d, k=1) or [""])[0][:EXC]])
        chiso.append((q, d))
    if i % 200 == 0: print(f"  băm {i}/{len(nhom)} | {(time.time()-t0)/60:.1f} phút", flush=True)
print(f"{len(cap):,} cặp · {(time.time()-t0)/60:.1f} phút")


In [ ]:
m = load_reranker("AITeamVN/Vietnamese_Reranker", device="cuda", max_length=512)
t0 = time.time(); sc = m.predict(cap)
tro = {}
for (q, d), v in zip(chiso, sc): tro.setdefault(q, {})[d] = float(v)
print(f"chấm xong · {(time.time()-t0)/60:.1f} phút · {(time.time()-t0)/len(cap):.3f} s/cặp")
json.dump(tro, open(f"{OUT}/diem_tro_600nhom.json", "w", encoding="utf-8"), ensure_ascii=False)


In [ ]:
# ===== So sánh =====
G = {q: {str(a) for a in train[q]["answer"]} for q in nhom}
top1 = lambda sc, q: max(sc[q], key=sc[q].get)
inK  = lambda sc, q, k: bool(G[q] & set(sorted(sc[q], key=lambda d: -sc[q][d])[:k]))

n = len(nhom)
tt1 = sum(top1(nhan, q) in G[q] for q in nhom) / n
ht1 = sum(top1(tro,  q) in G[q] for q in nhom) / n
tt2 = sum(inK(nhan, q, 2) for q in nhom) / n
ht2 = sum(inK(tro,  q, 2) for q in nhom) / n

print(f"{'':12}{'gold@1':>10}{'gold@2':>10}")
print(f"{'học trò':<12}{ht1:>10.4f}{ht2:>10.4f}")
print(f"{'THẦY':<12}{tt1:>10.4f}{tt2:>10.4f}")
print(f"{'Δ':<12}{tt1-ht1:>+10.4f}{tt2-ht2:>+10.4f}")

# thầy sửa được bao nhiêu lỗi của học trò, và làm hỏng bao nhiêu chỗ học trò đang đúng
tro_sai = [q for q in nhom if top1(tro, q) not in G[q]]
tro_dung = [q for q in nhom if top1(tro, q) in G[q]]
cuu  = sum(top1(nhan, q) in G[q] for q in tro_sai)
hong = sum(top1(nhan, q) not in G[q] for q in tro_dung)
print(f"\nhọc trò SAI {len(tro_sai)} nhóm → thầy CỨU {cuu} ({cuu/max(len(tro_sai),1):.1%})")
print(f"học trò ĐÚNG {len(tro_dung)} nhóm → thầy LÀM HỎNG {hong} ({hong/max(len(tro_dung),1):.1%})")
print(f"NET: {cuu - hong:+d} nhóm")

dat = (tt1 - ht1) >= 0.08
print(f"\nCỔNG (Δgold@1 >= +0,0800): {'✅ ĐẠT — chạy chungcat_run.ipynb' if dat else '❌ KHÔNG ĐẠT — thầy không biết thêm gì. DỪNG, đừng chưng cất.'}")
json.dump({"thay_g1": tt1, "tro_g1": ht1, "delta_g1": tt1-ht1,
           "cuu": cuu, "hong": hong, "net": cuu-hong, "dat_cong": bool(dat)},
          open(f"{OUT}/meta_congkiem.json", "w", encoding="utf-8"))
print("\nTẢI outputs/ VỀ TRƯỚC KHI ĐÓNG PHIÊN.")
